# 📈 Two-Step NARDL: Asymmetric Oil Price Pass-Through to Inflation

**Application:** Nonlinear ARDL estimation of the asymmetric effects of WTI crude oil prices on US Consumer Price Index (CPI), a canonical test of the **Shin, Yu & Greenwood-Nimmo (2014)** and **Cho, Greenwood-Nimmo & Shin (2019)** frameworks.

> *"Oil price increases raise consumer prices but oil price decreases do not fully feed through — a classic asymmetry."*

---

**Data:** FRED (Federal Reserve Economic Data), monthly, 1990–2024
- **Dependent variable:** `lcpi` — log US CPI All Items (CPIAUCSL)
- **Asymmetric variable:** `loil` — log WTI crude oil price (DCOILWTICO)

**Model:** $ \Delta \ln CPI_t = \rho \hat{u}_{t-1} + \sum_j \pi_j^+ \Delta \ln OIL_{t-j}^+ + \sum_j \pi_j^- \Delta \ln OIL_{t-j}^- + \varepsilon_t $

In [ ]:
# ============================================================
# 0. Setup
# ============================================================
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import rcParams

# --- Publication style ---
plt.style.use('seaborn-v0_8-whitegrid')
rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'figure.dpi': 110,
    'axes.spines.top': False,
    'axes.spines.right': False,
})
BLUE = '#1a6faf'
RED  = '#b22222'
GREEN = '#2e8b57'
GOLD  = '#cc8800'

print('Libraries loaded ✓')

## 1. Data Download & Preparation

In [ ]:
# ============================================================
# 1A. Download from FRED via pandas_datareader
# ============================================================
import pandas_datareader.data as web

START = '1990-01-01'
END   = '2024-06-01'

print('Downloading data from FRED...')
try:
    # US CPI All Urban Consumers (NSA, monthly)
    cpi = web.DataReader('CPIAUCSL', 'fred', START, END)
    # WTI Crude Oil Price (daily -> resample to monthly)
    oil_d = web.DataReader('DCOILWTICO', 'fred', START, END)
    oil = oil_d.resample('MS').mean()  # monthly average
    print(f'  CPI: {len(cpi)} months  |  Oil: {len(oil)} months')
    DATA_OK = True
except Exception as e:
    print(f'FRED download failed: {e}')
    DATA_OK = False

In [ ]:
# ============================================================
# 1B. Fallback: Try yfinance for oil, FRED for CPI
# ============================================================
if not DATA_OK:
    import yfinance as yf
    print('Trying yfinance fallback for oil...')
    oil_yf = yf.download('CL=F', start=START, end=END, progress=False)['Close']
    oil = oil_yf.resample('MS').mean().rename('DCOILWTICO')
    oil = pd.DataFrame(oil)
    print(f'  Oil from yfinance: {len(oil)} months')
    # For CPI: use a synthetic approximation for demonstration
    try:
        cpi = web.DataReader('CPIAUCSL', 'fred', START, END)
    except:
        print('Using synthetic CPI (demo mode)')
        rng = np.random.default_rng(42)
        dates = pd.date_range(START, END, freq='MS')
        cpi_vals = np.cumsum(rng.normal(0.002, 0.002, len(dates))) + 4.5
        cpi = pd.DataFrame({'CPIAUCSL': np.exp(cpi_vals)}, index=dates)
    DATA_OK = True

In [ ]:
# ============================================================
# 1C. Merge, log-transform, clean
# ============================================================
df_raw = pd.concat([cpi.rename(columns={'CPIAUCSL': 'cpi'}),
                    oil.rename(columns={'DCOILWTICO': 'oil'})], axis=1)
df_raw = df_raw.dropna()
df_raw.index = pd.to_datetime(df_raw.index)

df = pd.DataFrame(index=df_raw.index)
df['lcpi'] = np.log(df_raw['cpi'])
df['loil'] = np.log(df_raw['oil'])
df = df.dropna()

print(f'Sample: {df.index[0].strftime("%Y-%m")} — {df.index[-1].strftime("%Y-%m")}')
print(f'Observations: {len(df)}')
df.tail(3)

## 2. Exploratory Data Analysis

In [ ]:
# ============================================================
# 2A. Level and first-difference plots
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
fig.suptitle('WTI Oil Price and US CPI — Levels and First Differences', 
             fontsize=14, fontweight='bold', y=1.01)

# Levels
ax = axes[0, 0]
ax.plot(df.index, df['lcpi'], color=BLUE, lw=1.8, label='Log CPI')
ax.set_title('(a) Log CPI (levels)')
ax.set_ylabel('Log index')
ax2 = ax.twinx()
ax2.plot(df.index, df['loil'], color=RED, lw=1.2, alpha=0.6, label='Log Oil')
ax2.set_ylabel('Log price', color=RED)
ax.legend(loc='upper left'); ax2.legend(loc='upper right')

ax = axes[0, 1]
ax.fill_between(df.index, df['loil'].min(), df['loil'], alpha=0.15, color=RED)
ax.plot(df.index, df['loil'], color=RED, lw=1.5, label='Log WTI')
# Shade recessions (approx)
for start, end in [('2001-03', '2001-11'), ('2007-12', '2009-06'), 
                   ('2020-02', '2020-04')]:
    ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), alpha=0.12, color='gray')
ax.set_title('(b) Log WTI Crude Oil Price (NBER recessions shaded)')
ax.set_ylabel('Log USD/barrel')
ax.legend()

# Differences
dlcpi = df['lcpi'].diff().dropna() * 100
dloil = df['loil'].diff().dropna() * 100

ax = axes[1, 0]
ax.bar(dlcpi.index, dlcpi.values, color=BLUE, alpha=0.7, width=20)
ax.axhline(0, color='k', lw=0.8)
ax.set_title('(c) Month-on-Month CPI Inflation (%)')
ax.set_ylabel('%')

ax = axes[1, 1]
pos_dloil = np.where(dloil.values > 0, dloil.values, 0)
neg_dloil = np.where(dloil.values < 0, dloil.values, 0)
ax.bar(dloil.index, pos_dloil, color=RED, alpha=0.7, width=20, label='Increase')
ax.bar(dloil.index, neg_dloil, color=BLUE, alpha=0.7, width=20, label='Decrease')
ax.axhline(0, color='k', lw=0.8)
ax.set_title('(d) ΔLog WTI (% change, pos/neg shaded)')
ax.set_ylabel('%')
ax.legend()

plt.tight_layout()
plt.savefig('figures/fig1_raw_data.png', dpi=150, bbox_inches='tight')
plt.show()
print('Figure 1 saved.')

In [ ]:
# ============================================================
# 2B. Descriptive statistics
# ============================================================
from scipy import stats as scipy_stats

desc = pd.concat([df.describe().T, 
                  pd.Series({'lcpi': scipy_stats.jarque_bera(df['lcpi'])[1],
                             'loil': scipy_stats.jarque_bera(df['loil'])[1]},
                            name='JB p-val').to_frame()], axis=1)

print('\n=== Descriptive Statistics ===')
print(df.describe().round(4).to_string())

print('\n=== Correlation Matrix ===')
print(df.corr().round(4))

print('\n=== First Differences ===')
df_d = df.diff().dropna()
print(df_d.describe().round(4))

In [ ]:
# ============================================================
# 2C. Partial Sum Decomposition — Preview
# ============================================================
from twostep_nardl import partial_sums

oil_pos, oil_neg = partial_sums(df['loil'].values)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
fig.suptitle('Partial Sum Decomposition of Log WTI Oil Price', 
             fontsize=13, fontweight='bold')

ax = axes[0]
ax.plot(df.index, df['loil'], color='#888888', lw=1, alpha=0.6, label='Log Oil (levels)')
ax.plot(df.index, oil_pos, color=RED, lw=2, label='$X_t^+$ (pos. partial sum)')
ax.plot(df.index, oil_neg, color=BLUE, lw=2, label='$X_t^-$ (neg. partial sum)')
ax.set_title('(a) Positive and Negative Partial Sums')
ax.legend()
ax.set_ylabel('Log units')

ax = axes[1]
dloil = df['loil'].diff().dropna()
dx_pos = np.maximum(dloil.values, 0)
dx_neg = np.minimum(dloil.values, 0)
ax.fill_between(dloil.index, 0, dx_pos, color=RED, alpha=0.65, label='$\\Delta X^+$ (oil increases)')
ax.fill_between(dloil.index, 0, dx_neg, color=BLUE, alpha=0.65, label='$\\Delta X^-$ (oil decreases)')
ax.axhline(0, color='k', lw=0.8)
ax.set_title('(b) Period-by-Period Positive/Negative Changes')
ax.legend()
ax.set_ylabel('Δ Log units')

plt.tight_layout()
plt.savefig('figures/fig2_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Unit Root Pre-Testing

NARDL requires I(1) regressors (or I(0)). We verify integration orders using ADF and Zivot-Andrews.

In [ ]:
# ============================================================
# 3. ADF Unit Root Tests
# ============================================================
from statsmodels.tsa.stattools import adfuller

def adf_table(series_dict, lags='AIC'):
    rows = []
    for name, s in series_dict.items():
        res = adfuller(s.dropna(), autolag='AIC', regression='ct')
        rows.append({
            'Variable': name,
            'ADF stat': round(res[0], 3),
            'p-value': round(res[1], 4),
            'Lags': res[2],
            'CV 1%': round(res[4]['1%'], 3),
            'CV 5%': round(res[4]['5%'], 3),
            'Decision': 'I(0)' if res[1] < 0.05 else 'Non-stationary'
        })
    return pd.DataFrame(rows).set_index('Variable')

print('=== ADF Unit Root Tests (trend + constant) ===')
adf_levels = adf_table({
    'lcpi (level)': df['lcpi'],
    'loil (level)': df['loil'],
    'Δlcpi (diff)': df['lcpi'].diff(),
    'Δloil (diff)': df['loil'].diff(),
})
print(adf_levels.to_string())
print('\nConclusion: lcpi ~ I(1), loil ~ I(1) → suitable for NARDL bounds test')

## 4. NARDL Estimation

We estimate the **Two-Step NARDL(p, q)** model:

**Step 1 (FM-OLS):** Long-run relation
$$ \ln CPI_t = \alpha + \beta^+ \ln OIL_t^+ + \beta^- \ln OIL_t^- + u_t $$

**Step 2 (OLS):** Short-run ECM
$$ \Delta \ln CPI_t = \rho \hat{u}_{t-1} + \sum_{j=1}^{p-1} \phi_j \Delta \ln CPI_{t-j} + \sum_{j=0}^{q-1}(\pi_j^+ \Delta \ln OIL_{t-j}^+ + \pi_j^- \Delta \ln OIL_{t-j}^-) + \varepsilon_t $$

In [ ]:
# ============================================================
# 4A. Main estimation — automatic lag selection (BIC)
# ============================================================
from twostep_nardl import TwoStepNARDL

model = TwoStepNARDL(
    data      = df,
    depvar    = 'lcpi',
    xvars     = ['loil'],
    decompose = ['loil'],      # asymmetric decomposition of oil
    lags      = None,          # auto-select via BIC
    maxlags   = 6,
    method    = 'twostep',
    step1     = 'fmols',       # FM-OLS long-run step
    case      = 3,             # unrestricted intercept, no trend
    ic        = 'bic',
)

print('Estimating Two-Step NARDL (BIC lag selection)...')
res = model.fit()
print('Done!')
print(f'Selected lags: {res.lags}')
print(f'BIC value: {res.ic_opt:.4f}')

In [ ]:
# ============================================================
# 4B. Print full estimation results
# ============================================================
print(res.summary())

In [ ]:
# ============================================================
# 4C. Also estimate with fixed lags = [3, 3] for robustness
# ============================================================
model_fixed = TwoStepNARDL(
    data=df, depvar='lcpi', xvars=['loil'],
    decompose=['loil'], lags=[3, 3],
    method='twostep', step1='fmols', case=3,
)
res_fixed = model_fixed.fit()
print('NARDL(3,3) results:')
print(res_fixed.summary())

## 5. PSS Bounds Test for Cointegration

In [ ]:
# ============================================================
# 5A. Bounds test with full CV table
# ============================================================
from twostep_nardl.postestimation import bounds_test

bt = bounds_test(res, sig_levels=[0.10, 0.05, 0.01])

In [ ]:
# ============================================================
# 5B. Display the full PSS critical value table as a DataFrame
# ============================================================
from twostep_nardl import pss_cv_table, bounds_cv_table

print('\n=== PSS (2001) Asymptotic Critical Values — F-statistic, Case III ===')
tbl_F = pss_cv_table(case=3, stat='F')
print(tbl_F.round(2).to_string())

print('\n=== PSS (2001) Asymptotic Critical Values — t-statistic, Case III ===')
tbl_t = pss_cv_table(case=3, stat='t')
print(tbl_t.round(2).to_string())

print(f'\n=== CVs applicable to THIS model (k=1) ===')
cv_k1 = bounds_cv_table(k=1, case=3, stat='F', sig_levels=[0.10, 0.05, 0.025, 0.01])
print(cv_k1.round(3).to_string())

In [ ]:
# ============================================================
# 5C. Visual bounds test plot
# ============================================================
from twostep_nardl import bounds_cv_table

cv = bounds_cv_table(k=1, case=3, stat='F', sig_levels=[0.10, 0.05, 0.01])
fstat = res.F_pss

fig, ax = plt.subplots(figsize=(9, 5))

levels = ['10%', '5%', '1%']
col_colors = ['#ffe0b2', '#ffb74d', '#e65100']

for i, (lvl, col) in enumerate(zip(levels, col_colors)):
    ci0 = float(cv.loc[lvl, 'I(0)'])
    ci1 = float(cv.loc[lvl, 'I(1)'])
    ax.barh(i, ci1 - ci0, left=ci0, color=col, alpha=0.65, height=0.4,
            label=f'{lvl}: [{ci0:.2f}, {ci1:.2f}]')
    ax.text(ci0 - 0.2, i, f'I(0)={ci0:.2f}', ha='right', va='center', fontsize=9)
    ax.text(ci1 + 0.1, i, f'I(1)={ci1:.2f}', ha='left',  va='center', fontsize=9)

# F-statistic line
ax.axvline(fstat, color='navy', lw=2.5, linestyle='-', zorder=5,
           label=f'F-statistic = {fstat:.3f}')

ax.set_yticks(range(len(levels)))
ax.set_yticklabels(levels, fontsize=11)
ax.set_xlabel('F-statistic value', fontsize=11)
ax.set_title('PSS (2001) Bounds Test — F-Statistic vs Critical Value Bands\n'
             f'(k=1, Case III, Asymptotic) | Decision: {bt["decisions"][0.05].upper()}',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)

plt.tight_layout()
plt.savefig('figures/fig3_bounds_test.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Asymmetry Tests (Wald)

In [ ]:
# ============================================================
# 6. Wald tests for long-run and short-run asymmetry
# ============================================================
from twostep_nardl.postestimation import wald_test

wt = wald_test(res)

# Summary bar chart
fig, ax = plt.subplots(figsize=(8, 4))

labels = ['Long-Run\nAsymmetry', 'Short-Run\n(Additive)', 'Short-Run\n(Impact)']
Wvals = [res.W_lr, res.W_sr, res.W_impact]
pvals = [res.p_lr, res.p_sr, res.p_impact]
colors = [RED if p < 0.05 else BLUE for p in pvals]

bars = ax.bar(labels, Wvals, color=colors, alpha=0.75, edgecolor='white', width=0.5)

for bar, W, p in zip(bars, Wvals, pvals):
    stars = '***' if p < 0.01 else '**' if p < 0.05 else '*' if p < 0.10 else 'n.s.'
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'W={W:.2f}\n{stars}', ha='center', va='bottom', fontsize=10, fontweight='bold')

critical_05 = 3.84   # chi2(1) at 5%
ax.axhline(critical_05, color='k', lw=1.5, linestyle='--', label=f'χ²(1)@5% = {critical_05}')
ax.set_title('Wald Tests for Asymmetry\n(red bars = reject H₀: symmetry at 5%)', fontweight='bold')
ax.set_ylabel('Wald statistic (χ²)')
ax.legend()

plt.tight_layout()
plt.savefig('figures/fig4_wald_tests.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Long-Run Coefficients Comparison

In [ ]:
# ============================================================
# 7. Long-run coefficients: β+ vs β-
# ============================================================
from scipy import stats as scipy_stats

b_pos = float(res.b_lr[0])
b_neg = float(res.b_lr[1])
se_pos = float(np.sqrt(res.V_lr[0, 0]))
se_neg = float(np.sqrt(res.V_lr[1, 1]))
ci95_pos = (b_pos - 1.96*se_pos, b_pos + 1.96*se_pos)
ci95_neg = (b_neg - 1.96*se_neg, b_neg + 1.96*se_neg)

print(f'  Long-run β+ (oil increase) = {b_pos:.4f}  SE={se_pos:.4f}  95% CI=[{ci95_pos[0]:.4f}, {ci95_pos[1]:.4f}]')
print(f'  Long-run β- (oil decrease) = {b_neg:.4f}  SE={se_neg:.4f}  95% CI=[{ci95_neg[0]:.4f}, {ci95_neg[1]:.4f}]')
print(f'  Asymmetry: β+ - β- = {b_pos - b_neg:.4f}')
print(f'  Speed of adjustment (ρ) = {res.rho:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Coefficient comparison
ax = axes[0]
y_pos = [0, 1]
coefs = [b_pos, b_neg]
ses = [se_pos, se_neg]
colors_lr = [RED, BLUE]
labels_lr = ['β⁺ (oil increase)', 'β⁻ (oil decrease)']

ax.barh(y_pos, coefs, xerr=[1.96*s for s in ses],
        color=colors_lr, alpha=0.75, height=0.4,
        error_kw={'capsize': 5, 'linewidth': 2})
ax.axvline(0, color='k', lw=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(labels_lr, fontsize=11)
ax.set_xlabel('Coefficient value', fontsize=11)
ax.set_title('Long-Run Elasticities with 95% CI\n(FM-OLS Estimation)', fontweight='bold')

for y, c, s, l in zip(y_pos, coefs, ses, labels_lr):
    ax.text(c + 0.001, y + 0.07, f'{c:.4f}', ha='left', fontsize=10, fontweight='bold')

# Scatter: LR vs SR impact
ax = axes[1]
ax.set_title('Long-Run vs Short-Run Asymmetry\n', fontweight='bold')

# Extract SR impacts
sr_names = res.sr_names
sr_coefs = res.b_sr
sr_se = np.sqrt(np.diag(res.V_sr))

pos_mask = ['+' in n or 'pos' in n.lower() for n in sr_names]
neg_mask = ['-' in n or 'neg' in n.lower() for n in sr_names]

for i, (nm, b_, se_) in enumerate(zip(sr_names, sr_coefs, sr_se)):
    if 'pos' in nm.lower():
        ax.scatter(i, b_, color=RED, s=80, zorder=3)
        ax.errorbar(i, b_, yerr=1.96*se_, color=RED, fmt='none', capsize=4)
        ax.text(i, b_+0.0005, nm.replace('pos','⁺').replace('D.','Δ'), 
                ha='center', va='bottom', fontsize=7, color=RED)
    elif 'neg' in nm.lower():
        ax.scatter(i, b_, color=BLUE, s=80, zorder=3)
        ax.errorbar(i, b_, yerr=1.96*se_, color=BLUE, fmt='none', capsize=4)
        ax.text(i, b_-0.0005, nm.replace('neg','⁻').replace('D.','Δ'), 
                ha='center', va='top', fontsize=7, color=BLUE)
    elif 'ect' in nm.lower() or 'L.ect' in nm:
        ax.scatter(i, b_, color=GOLD, s=100, marker='D', zorder=4)
        ax.text(i, b_-0.0005, f'ρ={b_:.3f}', ha='center', va='top', fontsize=8, color=GOLD)
ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel('Coefficient')
ax.set_xlabel('Regressor index')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=RED, label='Δ Oil⁺ (increase)'),
                   Patch(color=BLUE, label='Δ Oil⁻ (decrease)'),
                   Patch(color=GOLD, label='Error correction (ρ)')])

plt.tight_layout()
plt.savefig('figures/fig5_lr_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Cumulative Dynamic Multipliers

In [ ]:
# ============================================================
# 8A. Compute and display multipliers
# ============================================================
from twostep_nardl.postestimation import multipliers

HORIZON = 48  # 4 years
mult = multipliers(res, horizon=HORIZON)

In [ ]:
# ============================================================
# 8B. Publication-quality multiplier plot
# ============================================================
h = np.arange(1, HORIZON + 1)
mp = mult['pos'].values
mn = mult['neg'].values
md = mult['diff'].values

lr_p = float(res.b_lr[0])
lr_n = float(res.b_lr[1])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Cumulative Dynamic Multipliers — Asymmetric Oil Price Pass-Through to CPI',
             fontsize=13, fontweight='bold')

# Panel A: Pos and Neg multipliers
ax = axes[0]
ax.fill_between(h, 0, md, alpha=0.12, color=GREEN, label='_nolegend_')
ax.plot(h, mp, color=RED, lw=2.5, label=f'Positive shock (oil ↑)  β⁺={lr_p:.4f}')
ax.plot(h, mn, color=BLUE, lw=2.5, label=f'Negative shock (oil ↓)  β⁻={lr_n:.4f}')
ax.axhline(lr_p, color=RED, lw=1, linestyle=':', alpha=0.6)
ax.axhline(lr_n, color=BLUE, lw=1, linestyle=':', alpha=0.6)
ax.axhline(0, color='k', lw=0.8)
ax.set_xlabel('Horizon (months)')
ax.set_ylabel('Cumulative response of log CPI')
ax.set_title('(a) Positive vs Negative Multipliers\n(dotted lines = LR equilibrium values)')
ax.legend(loc='lower right')

# Panel B: Asymmetry profile
ax = axes[1]
ax.fill_between(h, 0, md, alpha=0.25, color=GREEN)
ax.plot(h, md, color=GREEN, lw=2.5, label='Asymmetry (β⁺ − β⁻)')
ax.axhline(lr_p - lr_n, color=GREEN, lw=1, linestyle=':', alpha=0.7,
           label=f'LR asymmetry = {lr_p-lr_n:.4f}')
ax.axhline(0, color='k', lw=0.8)
# Mark half-life
from twostep_nardl.postestimation import half_life as hl_fn
hl_info = hl_fn(res, horizon=60, show_table=False)
if 'half_life' in hl_info:
    ax.axvline(int(hl_info['half_life']), color=GOLD, lw=1.5, linestyle='--',
               alpha=0.8, label=f'Half-life ≈ {int(hl_info["half_life"])} months')
ax.set_xlabel('Horizon (months)')
ax.set_ylabel('Asymmetry = m⁺(h) − m⁻(h)')
ax.set_title('(b) Cumulative Asymmetry Profile')
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('figures/fig6_multipliers.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Half-Life & Persistence Profile

In [ ]:
# ============================================================
# 9A. Half-life and persistence profile
# ============================================================
from twostep_nardl.postestimation import half_life

hl = half_life(res, horizon=72)

In [ ]:
# ============================================================
# 9B. Persistence profile plot
# ============================================================
pp = hl['pp_series'].values
h_pp = np.arange(len(pp))
pp_hl = hl.get('pp_halflife', np.nan)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Persistence Profile and Half-Life Analysis', 
             fontsize=13, fontweight='bold')

# Persistence profile
ax = axes[0]
ax.plot(h_pp, pp, color=BLUE, lw=2.5, label='PP(h) — Pesaran & Shin (1996)')
ax.fill_between(h_pp, 0, pp, alpha=0.1, color=BLUE)
ax.axhline(0.5, color=RED, lw=1.5, linestyle='--', alpha=0.8, label='50% threshold')
ax.axhline(0, color='k', lw=0.8)
if not np.isnan(pp_hl):
    ax.axvline(pp_hl, color=RED, lw=1.2, linestyle=':', alpha=0.8)
    ax.annotate(f'PP half-life = {pp_hl} months',
                xy=(pp_hl, 0.5), xytext=(pp_hl + 3, 0.62),
                fontsize=9.5, color='darkred',
                arrowprops=dict(arrowstyle='->', color='darkred'))
ax.set_xlabel('Horizon (months)')
ax.set_ylabel('Proportion of disequilibrium remaining')
ax.set_title('(a) Persistence Profile of ECM')
ax.set_ylim(-0.1, 1.15)
ax.legend()

# Adjustment speed interpretation
ax = axes[1]
rho = res.rho
t_arr = np.arange(0, 73)
impulse_resp = (1 + rho) ** t_arr

ax.fill_between(t_arr, 0, impulse_resp, alpha=0.12, color=GREEN)
ax.plot(t_arr, impulse_resp, color=GREEN, lw=2.5, label=f'$(1+\\rho)^t$,  ρ={rho:.4f}')
ax.axhline(0.5, color=RED, lw=1.2, linestyle='--', alpha=0.7, label='50% correction')
ax.axhline(0, color='k', lw=0.8)
if 'half_life' in hl:
    hl_val = hl['half_life']
    ax.axvline(hl_val, color=RED, lw=1.2, linestyle=':', alpha=0.8)
    ax.text(hl_val + 0.5, 0.55, f'HL = {hl_val:.1f}m', color='darkred', fontsize=9.5)
ax.set_xlabel('Horizon (months)')
ax.set_ylabel('Fraction of shock remaining')
ax.set_title(f'(b) Geometric Decay Curve\n(simple ECM impulse response, ρ={rho:.4f})')
ax.set_ylim(-0.1, 1.15)
ax.legend()

plt.tight_layout()
plt.savefig('figures/fig7_halflife.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Impulse Response Functions

In [ ]:
# ============================================================
# 10. Period-by-period IRF
# ============================================================
from twostep_nardl.postestimation import irf

irf_df = irf(res, horizon=36, show_table=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('Impulse Response Functions — 1% Oil Price Shock → CPI',
             fontsize=13, fontweight='bold')

for ax, col, color, lbl, panel in [
    (axes[0], 'pos_period', RED,   'Positive shock (oil↑)', '(a)'),
    (axes[1], 'neg_period', BLUE,  'Negative shock (oil↓)', '(b)'),
    (axes[2], 'pos_cum',    RED,   'Cumulative positive',   '(c)'),
]:
    h = np.arange(1, len(irf_df) + 1)
    vals = irf_df[col].values
    if 'period' in col:
        ax.bar(h, vals, color=color, alpha=0.7, edgecolor='white', width=0.8)
        if col == 'neg_period':
            ax2 = ax.twinx()
            ax2.plot(h, irf_df['neg_cum'].values, color='navy', lw=1.5, linestyle='-.',
                     label='Cumulative (right axis)')
            ax2.set_ylabel('Cumulative', color='navy')
            ax2.legend(loc='upper right', fontsize=8)
    else:
        ax.plot(h, vals, color=color, lw=2.2)
        ax.fill_between(h, 0, vals, alpha=0.12, color=color)
        ax.plot(h, irf_df['neg_cum'].values, color=BLUE, lw=2.2,
                linestyle='--', label='Cumulative negative')
        ax.legend(fontsize=9)
    ax.axhline(0, color='k', lw=0.8)
    ax.set_title(f'{panel} {lbl}')
    ax.set_xlabel('Horizon (months)')
    ax.set_ylabel('Response')

plt.tight_layout()
plt.savefig('figures/fig8_irf.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Residual Diagnostics

In [ ]:
# ============================================================
# 11A. Formal diagnostic tests
# ============================================================
from twostep_nardl.postestimation import diagnostics

diag = diagnostics(res)

In [ ]:
# ============================================================
# 11B. Residual plots
# ============================================================
from twostep_nardl.postestimation import predict

yhat = predict(res, kind='xb')
resid = predict(res, kind='residuals')

# Drop NaN
Dy_act = df['lcpi'].diff().dropna()
resid_nonan = resid.dropna()
yhat_nonan = yhat.dropna()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle('Residual Diagnostics — ECM Short-Run Regression', 
             fontsize=13, fontweight='bold')

# 1. Actual vs Fitted
ax = axes[0, 0]
idx_overlap = resid_nonan.index
dy_at_idx = Dy_act.reindex(idx_overlap)
ax.scatter(yhat_nonan.reindex(idx_overlap), dy_at_idx,
           alpha=0.4, s=15, color=BLUE, edgecolors='none')
mn_v = min(yhat_nonan.min(), dy_at_idx.min())
mx_v = max(yhat_nonan.max(), dy_at_idx.max())
ax.plot([mn_v, mx_v], [mn_v, mx_v], 'k--', lw=1.2)
ax.set_title('(a) Actual vs Fitted ΔlnCPI')
ax.set_xlabel('Fitted'); ax.set_ylabel('Actual')

# 2. Residual time series
ax = axes[0, 1]
ax.plot(resid_nonan.index, resid_nonan.values, color=BLUE, lw=1, alpha=0.8)
ax.axhline(0, color='k', lw=0.8)
ax.fill_between(resid_nonan.index,
                -2*resid_nonan.std(), 2*resid_nonan.std(),
                alpha=0.1, color='gray', label='±2σ')
ax.set_title('(b) Residuals over Time')
ax.set_ylabel('Residual')
ax.legend(fontsize=9)

# 3. QQ plot
ax = axes[0, 2]
from scipy.stats import probplot
probplot(resid_nonan.dropna().values, plot=ax)
ax.get_lines()[0].set(color=BLUE, markersize=3, alpha=0.5)
ax.get_lines()[1].set(color='k', lw=1.5)
ax.set_title('(c) Q-Q Plot of Residuals')

# 4. Residual histogram
ax = axes[1, 0]
resid_vals = resid_nonan.dropna().values
ax.hist(resid_vals, bins=40, color=BLUE, alpha=0.7, density=True, edgecolor='white')
xr = np.linspace(resid_vals.min(), resid_vals.max(), 200)
from scipy.stats import norm
ax.plot(xr, norm.pdf(xr, resid_vals.mean(), resid_vals.std()), 
        color=RED, lw=2, label='Normal PDF')
ax.set_title('(d) Residual Distribution')
ax.legend()

# 5. ACF of residuals
ax = axes[1, 1]
from statsmodels.graphics.tsaplots import plot_acf
plot_acf(resid_vals, lags=24, ax=ax, alpha=0.05, color=BLUE,
         vlines_kwargs={'color': BLUE})
ax.set_title('(e) ACF of Residuals')

# 6. Squared residuals ACF (ARCH test)
ax = axes[1, 2]
plot_acf(resid_vals**2, lags=24, ax=ax, alpha=0.05, color=RED,
         vlines_kwargs={'color': RED})
ax.set_title('(f) ACF of Squared Residuals (ARCH)')

plt.tight_layout()
plt.savefig('figures/fig9_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. CUSUM Stability Test

In [ ]:
# ============================================================
# 12. CUSUM structural stability test
# ============================================================
from twostep_nardl.plotting import plot_cusum

fig = plot_cusum(res, title='CUSUM Stability Test — SR ECM Residuals', show=False)
plt.savefig('figures/fig10_cusum.png', dpi=150, bbox_inches='tight')
plt.show()

## 13. ECM Parameter Table (Publication Style)

In [ ]:
# ============================================================
# 13. Full publication table (CGS 2019 style)
# ============================================================
from twostep_nardl.postestimation import ecm_table

ecm_table(res)

## 14. Model Comparison: One-Step vs Two-Step

In [ ]:
# ============================================================
# 14. Compare one-step vs two-step
# ============================================================
model_os = TwoStepNARDL(
    data=df, depvar='lcpi', xvars=['loil'],
    decompose=['loil'], lags=res.lags,
    method='onestep', case=3,
)
res_os = model_os.fit()

print('\n' + '='*78)
print('  MODEL COMPARISON: Two-Step FM-OLS vs One-Step OLS')
print('='*78)

compare_df = pd.DataFrame({
    'Two-Step (FM-OLS)': {
        'β⁺ (LR pos.)': f"{float(res.b_lr[0]):.4f}",
        'β⁻ (LR neg.)': f"{float(res.b_lr[1]):.4f}",
        'ρ (ECM speed)': f"{float(res.rho):.4f}",
        'F-PSS': f"{float(res.F_pss):.3f}",
        'W-LR': f"{float(res.W_lr):.3f} (p={res.p_lr:.3f})",
        'R²': f"{float(res.r2):.4f}",
        'Adj. R²': f"{float(res.r2_a):.4f}",
        'Lags (p,q)': str(res.lags),
    },
    'One-Step (OLS)': {
        'β⁺ (LR pos.)': f"{float(res_os.b_lr[0]):.4f}",
        'β⁻ (LR neg.)': f"{float(res_os.b_lr[1]):.4f}",
        'ρ (ECM speed)': f"{float(res_os.rho):.4f}",
        'F-PSS': f"{float(res_os.F_pss):.3f}",
        'W-LR': f"{float(res_os.W_lr):.3f} (p={res_os.p_lr:.3f})",
        'R²': f"{float(res_os.r2):.4f}",
        'Adj. R²': f"{float(res_os.r2_a):.4f}",
        'Lags (p,q)': str(res_os.lags),
    },
})
print(compare_df.to_string())

In [ ]:
# ============================================================
# 14B. Visual comparison of LR estimates
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
fig.suptitle('Two-Step vs One-Step NARDL — Long-Run Coefficients',
             fontsize=13, fontweight='bold')

models_comp = [
    ('Two-Step\n(FM-OLS)', res, BLUE),
    ('One-Step\n(OLS)',    res_os, GREEN),
]

for ax, (lbl, rc, colo) in zip(axes, [(lbl, rc, colo) for lbl, rc, colo in
                                        [(m[0], m[1], m[2]) for m in models_comp]]):
    bp = float(rc.b_lr[0])
    bn = float(rc.b_lr[1])
    sep = float(np.sqrt(rc.V_lr[0, 0]))
    sen = float(np.sqrt(rc.V_lr[1, 1]))

    ax.barh([1, 0], [bp, bn], xerr=[1.96*sep, 1.96*sen],
            color=[RED, BLUE], alpha=0.75, height=0.4,
            error_kw={'capsize': 5, 'linewidth': 2})
    ax.axvline(0, color='k', lw=0.8)
    ax.set_yticks([0, 1])
    ax.set_yticklabels(['β⁻ (oil decrease)', 'β⁺ (oil increase)'], fontsize=11)
    ax.set_title(f'{lbl}\nβ⁺={bp:.4f}, β⁻={bn:.4f}')
    ax.set_xlabel('Coefficient (with 95% CI)')
    ax.text(bp, 1.22, f'{bp:.4f}', ha='center', fontsize=10, fontweight='bold', color=RED)
    ax.text(bn, -0.22, f'{bn:.4f}', ha='center', fontsize=10, fontweight='bold', color=BLUE)

plt.tight_layout()
plt.savefig('figures/fig11_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 15. Summary Dashboard

In [ ]:
# ============================================================
# 15. All-in-one results dashboard
# ============================================================
fig = plt.figure(figsize=(18, 12))
fig.suptitle(
    'Asymmetric Oil Price Pass-Through to US CPI\nTwo-Step NARDL Results Dashboard',
    fontsize=16, fontweight='bold', y=0.98
)
gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.45, wspace=0.35)

# — Panel 1: Data (full period) —
ax1 = fig.add_subplot(gs[0, :2])
ax1.plot(df.index, df['lcpi'], color=BLUE, lw=1.8, label='Log CPI')
ax1b = ax1.twinx()
ax1b.plot(df.index, df['loil'], color=RED, lw=1.2, alpha=0.55, label='Log WTI')
ax1.set_title('Log CPI & WTI Oil Price', fontsize=11, fontweight='bold')
ax1.set_ylabel('Log CPI', color=BLUE)
ax1b.set_ylabel('Log WTI', color=RED)
ax1.legend(loc='upper left', fontsize=9)
ax1b.legend(loc='lower right', fontsize=9)

# — Panel 2: Partial sums —
ax2 = fig.add_subplot(gs[0, 2:])
ax2.plot(df.index, oil_pos, color=RED, lw=1.6, label='$X_t^+$')
ax2.plot(df.index, oil_neg, color=BLUE, lw=1.6, label='$X_t^-$')
ax2.set_title('Partial Sum Decomposition of Log WTI', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)

# — Panel 3: Multipliers —
ax3 = fig.add_subplot(gs[1, :2])
h_m = np.arange(1, HORIZON + 1)
ax3.plot(h_m, mult['pos'].values, color=RED, lw=2, label='Positive shock')
ax3.plot(h_m, mult['neg'].values, color=BLUE, lw=2, label='Negative shock')
ax3.axhline(lr_p, color=RED, lw=0.8, linestyle=':', alpha=0.6)
ax3.axhline(lr_n, color=BLUE, lw=0.8, linestyle=':', alpha=0.6)
ax3.axhline(0, color='k', lw=0.8)
ax3.set_title('Cumulative Dynamic Multipliers', fontsize=11, fontweight='bold')
ax3.set_xlabel('Horizon (months)')
ax3.legend(fontsize=9)

# — Panel 4: Asymmetry —
ax4 = fig.add_subplot(gs[1, 2:])
ax4.plot(h_m, mult['diff'].values, color=GREEN, lw=2)
ax4.fill_between(h_m, 0, mult['diff'].values, alpha=0.2, color=GREEN)
ax4.axhline(0, color='k', lw=0.8)
ax4.axhline(lr_p - lr_n, color=GREEN, lw=1, linestyle=':', alpha=0.7)
ax4.set_title('Cumulative Asymmetry Profile', fontsize=11, fontweight='bold')
ax4.set_xlabel('Horizon (months)')

# — Panel 5: Bounds test chart —
ax5 = fig.add_subplot(gs[2, 0])
cvs = bounds_cv_table(k=1, case=3, stat='F', sig_levels=[0.10, 0.05, 0.01])
for i, (lvl, c) in enumerate(zip(['10%', '5%', '1%'],
                                   ['#ffe0b2', '#ffb74d', '#f57c00'])):
    ci0 = float(cvs.loc[lvl, 'I(0)'])
    ci1 = float(cvs.loc[lvl, 'I(1)'])
    ax5.barh(i, ci1-ci0, left=ci0, color=c, height=0.35)
ax5.axvline(res.F_pss, color='navy', lw=2, label=f'F={res.F_pss:.2f}')
ax5.set_yticks([0,1,2]); ax5.set_yticklabels(['10%', '5%', '1%'], fontsize=9)
ax5.set_title('Bounds Test', fontsize=11, fontweight='bold')
ax5.legend(fontsize=9)

# — Panel 6: Wald tests —
ax6 = fig.add_subplot(gs[2, 1])
wlabels = ['LR', 'SR\nAdditive', 'SR\nImpact']
wvals   = [res.W_lr, res.W_sr, res.W_impact]
wpvals  = [res.p_lr, res.p_sr, res.p_impact]
wcols   = [RED if p < 0.05 else BLUE for p in wpvals]
ax6.bar(wlabels, wvals, color=wcols, alpha=0.75)
ax6.axhline(3.84, color='k', lw=1.2, linestyle='--', label='5% CV')
ax6.set_title('Wald Asymmetry\nTests', fontsize=11, fontweight='bold')
ax6.legend(fontsize=9)

# — Panel 7: Persistence —
ax7 = fig.add_subplot(gs[2, 2])
ax7.plot(h_pp, pp, color=BLUE, lw=2)
ax7.axhline(0.5, color=RED, lw=1, linestyle='--', alpha=0.7)
ax7.axhline(0, color='k', lw=0.8)
if not np.isnan(pp_hl):
    ax7.axvline(pp_hl, color=RED, lw=1, linestyle=':', alpha=0.8)
ax7.set_title('Persistence Profile', fontsize=11, fontweight='bold')
ax7.set_xlabel('Months')

# — Panel 8: Key statistics text box —
ax8 = fig.add_subplot(gs[2, 3])
ax8.axis('off')
summary_text = (
    f"KEY RESULTS\n"
    f"{'─'*28}\n"
    f"Method: Two-Step FM-OLS\n"
    f"Lags: {res.lags}\n\n"
    f"Long-Run Elasticities:\n"
    f"  β⁺ = {float(res.b_lr[0]):.4f}\n"
    f"  β⁻ = {float(res.b_lr[1]):.4f}\n"
    f"  Asymmetry: {float(res.b_lr[0]-res.b_lr[1]):.4f}\n\n"
    f"ECM speed: ρ = {res.rho:.4f}\n"
    f"BDM t-stat = {res.t_bdm:.3f}\n\n"
    f"Bounds Test:\n"
    f"  F = {res.F_pss:.3f}\n"
    f"  Decision @ 5%: {bt['decisions'][0.05].upper()}\n\n"
    f"Goodness-of-Fit:\n"
    f"  R² = {res.r2:.4f}\n"
    f"  Adj.R² = {res.r2_a:.4f}\n"
    f"  N = {res.nobs:,}"
)
ax8.text(0.05, 0.97, summary_text, transform=ax8.transAxes,
         va='top', ha='left', fontsize=9.5, fontfamily='monospace',
         bbox=dict(boxstyle='round,pad=0.6', facecolor='#f5f5f5', edgecolor='#cccccc'))

plt.savefig('figures/fig12_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✅ Dashboard saved.')

## 16. References

1. **Cho, J.S., Greenwood-Nimmo, M. and Shin, Y. (2019).** *Two-step estimation of the nonlinear autoregressive distributed lag model.* Working Paper.

2. **Shin, Y., Yu, B. and Greenwood-Nimmo, M. (2014).** *Modelling asymmetric cointegration and dynamic multipliers in a nonlinear ARDL framework.* In: W. Horrace and R. Sickles (eds.), Festschrift in Honor of Peter Schmidt. Springer, New York.

3. **Pesaran, M.H., Shin, Y. and Smith, R.J. (2001).** *Bounds testing approaches to the analysis of level relationships.* Journal of Applied Econometrics, 16(3), 289–326.

4. **Kripfganz, S. and Schneider, D.C. (2020).** *Response surface regressions for critical value bounds and approximate p-values in equilibrium correction models.* Oxford Bulletin of Economics and Statistics, 82(6), 1456–1481.

5. **Pesaran, M.H. and Shin, Y. (1996).** *Cointegration and speed of convergence to equilibrium.* Journal of Econometrics, 71(1–2), 117–143.

---
*Generated with `twostep-nardl` v3.0.0 — https://github.com/merwan-roudane/twostep-nardl*